# Part 1 — Data Audit, EDA & Business Understanding
## D2C Customer Churn Intelligence

**Objective**: Understand the business problem and audit the raw data before building any model.

**Snapshot Date**: 2025-09-30

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path("data")
SNAPSHOT = pd.Timestamp("2025-09-30")

print("=" * 60)
print("LOADING ALL DATASETS")
print("=" * 60)

In [ ]:
# Load all raw datasets
customers = pd.read_csv(DATA_DIR / "customers.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
tickets = pd.read_csv(DATA_DIR / "support_tickets.csv")
events = pd.read_csv(DATA_DIR / "web_events_snapshot.csv")
labels = pd.read_csv(DATA_DIR / "churn_labels.csv")
campaigns = pd.read_csv(DATA_DIR / "intervention_history.csv")

datasets = {
    'customers': customers,
    'orders': orders,
    'support_tickets': tickets,
    'web_events': events,
    'churn_labels': labels,
    'intervention_history': campaigns
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(f"DATASET : {name.upper()}")
    print("=" * 60)
    print(f"Shape : {df.shape}")
    print("\nFirst 3 Records:")
    display(df.head(3))
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum())

---
## 1. Data Quality Report

In [ ]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# Convert dates
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
orders["order_date"] = pd.to_datetime(orders["order_date"])
tickets["ticket_date"] = pd.to_datetime(tickets["ticket_date"])

In [ ]:
# Missing values by dataset
print("\n" + "=" * 60)
print("MISSING VALUES BY DATASET")
print("=" * 60)

for name, df in datasets.items():
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"\n{name.upper()} Missing Values:")
        report = pd.DataFrame({
            'Missing_Count': missing[missing > 0],
            'Missing_Percent': round((missing[missing > 0] / len(df)) * 100, 2)
        }).sort_values('Missing_Count', ascending=False)
        display(report)
    else:
        print(f"\n{name.upper()}: No missing values")

In [ ]:
# Duplicate-like records in orders
print("\n" + "=" * 60)
print("DUPLICATE CHECK - ORDERS")
print("=" * 60)

order_ids = orders['order_id'].values
dup_orders = orders[orders['order_id'].str.contains('_DUP', na=False)]
print(f"Duplicate-like records (_DUP suffix): {len(dup_orders)}")

if len(dup_orders) > 0:
    display(dup_orders.head())
    
    # Compare with originals
    dup_orders = dup_orders.copy()
    dup_orders['original_order_id'] = dup_orders['order_id'].str.replace('_DUP', '', regex=False)
    original_orders = orders[orders['order_id'].isin(dup_orders['original_order_id'])].copy()
    comparison = dup_orders.merge(original_orders, left_on='original_order_id', right_on='order_id', suffixes=('_dup', '_orig'))
    print(f"\nMatched duplicate pairs: {len(comparison)}")
    
    cols_to_compare = ['customer_id', 'order_date', 'category', 'quantity', 'gross_amount', 'discount_pct', 'delivery_days', 'returned', 'rating']
    for col in cols_to_compare:
        comparison[f'{col}_match'] = (comparison[f'{col}_dup'] == comparison[f'{col}_orig'])
    
    match_cols = [c for c in comparison.columns if c.endswith('_match')]
    comparison['all_fields_match'] = comparison[match_cols].all(axis=1)
    
    total_dup = len(comparison)
    exact_dup = comparison['all_fields_match'].sum()
    print(f"\nTotal DUP Records: {total_dup}")
    print(f"Exact Duplicates: {exact_dup}")
    print(f"Non-Exact Duplicates: {total_dup - exact_dup}")

# Clean orders
orders_clean = orders[~orders['order_id'].str.contains('_DUP', na=False)].copy()
print(f"\nOrders before cleaning: {len(orders)}")
print(f"Orders after cleaning: {len(orders_clean)}")
print(f"Records removed: {len(orders) - len(orders_clean)}")

In [ ]:
# Leakage check
print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

orders_pre = orders[orders["order_date"] <= SNAPSHOT].copy()
orders_post = orders[orders["order_date"] > SNAPSHOT].copy()
print(f"Post-snapshot orders (would be leakage): {len(orders_post)}")
print(f"Pre-snapshot orders (valid for modeling): {len(orders_pre)}")

In [ ]:
# Invalid value checks
print("\n" + "=" * 60)
print("INVALID VALUE CHECKS")
print("=" * 60)

print(f"Negative gross amount: {(orders['gross_amount'] < 0).sum()}")
print(f"Invalid quantity (<=0): {(orders['quantity'] <= 0).sum()}")
print(f"Invalid rating (not in 1-5): {((~orders['rating'].between(1, 5)) & orders['rating'].notna()).sum()}")
print(f"Invalid sentiment (not in -1 to 1): {((~tickets['sentiment_score'].between(-1, 1))).sum()}")

In [ ]:
# Join validation - orphan records
print("\n" + "=" * 60)
print("JOIN VALIDATION (Orphan Records)")
print("=" * 60)

cust_ids = set(customers["customer_id"])
orders_orphans = len(set(orders["customer_id"]) - cust_ids)
ticket_orphans = len(set(tickets["customer_id"]) - cust_ids)
event_orphans = len(set(events["customer_id"]) - cust_ids)

print(f"Customers in system: {len(cust_ids)}")
print(f"Order orphans (customer_id not in customers): {orders_orphans}")
print(f"Ticket orphans (customer_id not in customers): {ticket_orphans}")
print(f"Event orphans (customer_id not in customers): {event_orphans}")

# Orders before signup
date_check = orders_pre.merge(customers[["customer_id", "signup_date"]], on="customer_id", how="left")
invalid_orders = date_check[date_check["order_date"] < date_check["signup_date"]]
print(f"\nOrders placed before customer signup date: {len(invalid_orders)}")

In [ ]:
# Outlier detection - Gross Amount (IQR method)
print("\n" + "=" * 60)
print("OUTLIER DETECTION - Gross Amount")
print("=" * 60)

Q1 = orders_clean['gross_amount'].quantile(0.25)
Q3 = orders_clean['gross_amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
gross_outliers = orders_clean[(orders_clean['gross_amount'] < lower_bound) | (orders_clean['gross_amount'] > upper_bound)]

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Upper Bound: {upper_bound:.2f}")
print(f"Number of Gross Amount Outliers: {len(gross_outliers)}")
display(gross_outliers[['order_id', 'customer_id', 'gross_amount']].sort_values('gross_amount', ascending=False).head(10))

# Outlier detection - Resolution Hours
print("\n" + "=" * 60)
print("OUTLIER DETECTION - Resolution Hours")
print("=" * 60)

Q1_r = tickets['resolution_hours'].quantile(0.25)
Q3_r = tickets['resolution_hours'].quantile(0.75)
IQR_r = Q3_r - Q1_r
upper_bound_r = Q3_r + 1.5 * IQR_r
resolution_outliers = tickets[tickets['resolution_hours'] > upper_bound_r]

print(f"Q1: {Q1_r:.2f}")
print(f"Q3: {Q3_r:.2f}")
print(f"IQR: {IQR_r:.2f}")
print(f"Upper Bound: {upper_bound_r:.2f}")
print(f"Number of Resolution Time Outliers: {len(resolution_outliers)}")
display(resolution_outliers[['ticket_id', 'resolution_hours']].sort_values('resolution_hours', ascending=False).head(10))

print(f"\nGross Amount Outlier %: {round(len(gross_outliers) / len(orders_clean) * 100, 2)}%")
print(f"Resolution Time Outlier %: {round(len(resolution_outliers) / len(tickets) * 100, 2)}%")

---
## 2. Feature Aggregation & Data Merging

In [ ]:
# Build aggregated feature table
print("\n" + "=" * 60)
print("BUILDING AGGREGATED FEATURE TABLE")
print("=" * 60)

order_agg = orders_pre.groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    total_spend=("gross_amount", "sum"),
    avg_order_value=("gross_amount", "mean"),
    avg_discount=("discount_pct", "mean"),
    return_rate=("returned", "mean"),
    avg_rating=("rating", "mean"),
    avg_delivery_days=("delivery_days", "mean")
).reset_index()

ticket_agg = tickets.groupby("customer_id").agg(
    ticket_count=("ticket_id", "count"),
    avg_resolution=("resolution_hours", "mean"),
    avg_sentiment=("sentiment_score", "mean"),
    reopened_rate=("reopened", "mean")
).reset_index()

# Merge into single dataframe
df = (customers
      .merge(order_agg, on="customer_id", how="left")
      .merge(ticket_agg, on="customer_id", how="left")
      .merge(events, on="customer_id", how="left")
      .merge(campaigns, on="customer_id", how="left")
      .merge(labels, on="customer_id", how="left"))

print(f"Merged dataset shape: {df.shape}")

# Fill missing values
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

print(f"\nChurn rate: {df['churn_next_60d'].mean():.2%}")
print(f"\nFeature columns: {df.columns.tolist()}")

---
## 3. Exploratory Analysis & Visualizations

In [ ]:
# FIGURE 1: Churn Distribution
plt.figure(figsize=(8, 5))
ax = sns.countplot(x="churn_next_60d", data=df, palette="viridis")
plt.title("Churn Distribution (Next 60 Days)", fontsize=14, fontweight='bold')
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width()/2., p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

print("\nChurn Distribution (%):")
print(df["churn_next_60d"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# FIGURE 2: City Tier Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(x="city_tier", data=df, ax=axes[0], palette="Set2")
axes[0].set_title("City Tier Distribution", fontsize=12, fontweight='bold')

sns.countplot(x="age_group", data=df, ax=axes[1], palette="Set2", order=['18-24','25-34','35-44','45+'])
axes[1].set_title("Age Group Distribution", fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

acquisition_order = df['acquisition_channel'].value_counts().index
sns.countplot(y="acquisition_channel", data=df, ax=axes[2], palette="Set2", order=acquisition_order)
axes[2].set_title("Acquisition Channel", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 3: Orders by Product Category & Discount Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Product category distribution
category_counts = orders_clean['category'].value_counts()
category_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Orders by Product Category', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=45)

# Discount distribution
sns.histplot(orders_clean['discount_pct'], bins=20, ax=axes[1], color='coral')
axes[1].set_title('Discount Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Discount %')

# Delivery time distribution
sns.boxplot(x=orders_clean['delivery_days'], ax=axes[2], color='lightgreen')
axes[2].set_title('Delivery Time Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

display(orders_clean['category'].value_counts().reset_index())
print("\nDiscount Stats:")
print(orders_clean['discount_pct'].describe())
print("\nDelivery Days Stats:")
print(orders_clean['delivery_days'].describe())

In [ ]:
# FIGURE 4: Customer Ratings & Returns
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ratings
orders_clean['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='gold')
axes[0].set_title('Customer Ratings', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Return rate
sns.countplot(x='returned', data=orders_clean, ax=axes[1], palette='Set1')
axes[1].set_title('Returned vs Non-Returned Orders', fontsize=12, fontweight='bold')

# Total orders histogram
sns.histplot(df["total_orders"], bins=30, ax=axes[2], color='purple')
axes[2].set_title("Total Orders Distribution", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

return_rate = orders_clean['returned'].mean() * 100
print(f"Overall Return Rate: {return_rate:.2f}%")
print(f"Rating Stats:\n{orders_clean['rating'].describe()}")

In [ ]:
# FIGURE 5: Support Tickets Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Issue types
tickets['issue_type'].value_counts().plot(kind='bar', ax=axes[0], color='teal')
axes[0].set_title('Support Ticket Issue Types', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Reopened tickets
sns.countplot(x='reopened', data=tickets, ax=axes[1], palette='Set2')
axes[1].set_title('Reopened Tickets', fontsize=12, fontweight='bold')

# Sentiment distribution
sns.histplot(tickets['sentiment_score'], bins=20, ax=axes[2], color='darkorange')
axes[2].set_title('Sentiment Score Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

reopened_rate = tickets['reopened'].mean() * 100
print(f"Reopened Ticket Rate: {reopened_rate:.2f}%")
display(tickets['issue_type'].value_counts().reset_index())

In [ ]:
# FIGURE 6: Behavioral Signals vs Churn
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sessions vs Churn
sns.boxplot(x="churn_next_60d", y="sessions_30d", data=df, ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Sessions (Last 30d) vs Churn", fontsize=12, fontweight='bold')

# Last visit vs Churn
sns.boxplot(x="churn_next_60d", y="last_visit_days_ago", data=df, ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Last Visit (Days Ago) vs Churn", fontsize=12, fontweight='bold')

# Total spend vs Churn
sns.boxplot(x="churn_next_60d", y="total_spend", data=df, ax=axes[1, 0], palette="Set2")
axes[1, 0].set_title("Total Spend vs Churn", fontsize=12, fontweight='bold')

# Total orders vs Churn
sns.boxplot(x="churn_next_60d", y="total_orders", data=df, ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("Total Orders vs Churn", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Churn-Risk Hypotheses (Evidence-Based)

In [ ]:
print("\n" + "=" * 70)
print("CHURN HYPOTHESIS ANALYSIS")
print("=" * 70)

# HYPOTHESIS 1: Lower engagement -> Higher churn
print("\n" + "=" * 70)
print("HYPOTHESIS 1: LOWER ENGAGEMENT → HIGHER CHURN")
print("=" * 70)
h1 = df.groupby("churn_next_60d")["sessions_30d"].mean().round(2).reset_index()
print(h1)
print("→ Interpretation: Customers who churn have significantly fewer sessions in the last 30 days."
      " Low engagement is a leading indicator of churn.")

# HYPOTHESIS 2: Longer inactivity -> Higher churn
print("\n" + "=" * 70)
print("HYPOTHESIS 2: LONGER INACTIVITY → HIGHER CHURN")
print("=" * 70)
h2 = df.groupby("churn_next_60d")["last_visit_days_ago"].mean().round(2).reset_index()
print(h2)
print("→ Interpretation: Churned customers have more days since their last visit."
      " Customers who haven't visited recently are at higher risk.")

# HYPOTHESIS 3: Lower purchase frequency -> Higher churn
print("\n" + "=" * 70)
print("HYPOTHESIS 3: LOWER PURCHASE FREQUENCY → HIGHER CHURN")
print("=" * 70)
h3 = df.groupby("churn_next_60d")["total_orders"].mean().round(2).reset_index()
print(h3)
print("→ Interpretation: Customers who churn have placed fewer orders overall."
      " Low purchase frequency signals weak product stickiness.")

# HYPOTHESIS 4: Lower wishlist activity -> Higher churn
print("\n" + "=" * 70)
print("HYPOTHESIS 4: LOWER WISHLIST ACTIVITY → HIGHER CHURN")
print("=" * 70)
h4 = df.groupby("churn_next_60d")["wishlist_adds_30d"].mean().round(2).reset_index()
print(h4)
print("→ Interpretation: Churned customers add fewer items to wishlists."
      " Lower intent-to-purchase signals are correlated with churn.")

# HYPOTHESIS 5: Higher support tickets -> Higher churn
print("\n" + "=" * 70)
print("HYPOTHESIS 5: HIGHER SUPPORT TICKETS → HIGHER CHURN")
print("=" * 70)
h5 = df.groupby("churn_next_60d")["ticket_count"].mean().round(2).reset_index()
print(h5)
print("→ Interpretation: Customers with more support tickets are more likely to churn."
      " Frequent complaints indicate dissatisfaction.")

---
## 5. Data Quality Summary

In [ ]:
print("\n" + "=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

issues = []

# Missing values
for name, ds in datasets.items():
    mv = ds.isnull().sum().sum()
    if mv > 0:
        issues.append(f"{name}: {mv} missing values ({round(mv/len(ds)*100,2)}%)")
    else:
        issues.append(f"{name}: No missing values")

# Duplicates
issues.append(f"Orders: {len(dup_orders)} _DUP records found and removed")

# Outliers
issues.append(f"Orders: {len(gross_outliers)} gross amount outliers ({round(len(gross_outliers)/len(orders_clean)*100,2)}%)")
issues.append(f"Tickets: {len(resolution_outliers)} resolution time outliers ({round(len(resolution_outliers)/len(tickets)*100,2)}%)")

# Invalid values
issues.append(f"Orders: {len(orders[orders['gross_amount']<0])} negative amounts")
issues.append(f"Orders: {len(orders[orders['quantity']<=0])} invalid quantities")

# Orphans
issues.append(f"Orphan orders: {orders_orphans} customer_ids not in customers table")
issues.append(f"Orders before signup: {len(invalid_orders)} order_date < signup_date")

# Leakage
issues.append(f"Leakage: {len(orders_post)} orders after snapshot date that were excluded from feature engineering")

for issue in issues:
    print(f"  • {issue}")

print("\n" + "=" * 70)
print("EDA COMPLETE")
print("=" * 70)